# Meteosat cloud animation around Maroantsetra flood dates

This notebook builds short cloud-motion animations around known flood/cyclone dates near Maroantsetra, Madagascar.

Default events included:
- Cyclone Herold / Maroantsetra flooding: 2020-03-13 to 2020-03-18.
- Cyclone Gamane / north-east Madagascar impacts: 2024-03-26 to 2024-03-29.

Add or edit events in the parameters cell if you have field-confirmed flood dates.

## 1. Install and imports

In [ ]:
!pip -q install earthengine-api geemap requests pillow

In [ ]:
import os
import math
import requests
import ee
import geemap
import pandas as pd
from IPython.display import display, Image, HTML

## 2. Parameters

In [ ]:
PROJECT_ID = "ee-rnrimpact"

MAROANTSETRA_LON = 49.7333
MAROANTSETRA_LAT = -15.4333
AOI_BUFFER_KM = 220

# Primary target: Meteosat Second Generation SEVIRI Indian Ocean Data Coverage.
# If your Earth Engine account cannot access this collection, the notebook can fall back
# to Oya precipitation, which is derived from geostationary VIS/IR sensors including Meteosat-9/10.
METEOSAT_COLLECTION_ID = "EO:EUM:DAT:MSG:HRSEVIRI-IODC"
FALLBACK_COLLECTION_ID = "projects/global-precipitation-nowcast/assets/global_estimation"
USE_FALLBACK_PRECIP_IF_METEOSAT_UNAVAILABLE = True

# Leave as None to auto-pick a likely cloud/IR band from the collection.
CLOUD_BAND = None
PREFERRED_CLOUD_BANDS = [
    "IR_108", "IR108", "IR_10_8", "IR_039", "IR_087", "IR_097", "IR_120", "IR_134",
    "WV_062", "WV_073", "VIS006", "VIS008", "HRV", "precipitation"
]

FLOOD_EVENTS = [
    {
        "name": "Cyclone Herold - Maroantsetra floods",
        "start": "2020-03-13T00:00:00",
        "end": "2020-03-18T23:59:59"
    },
    {
        "name": "Cyclone Gamane - north-east Madagascar floods",
        "start": "2024-03-26T00:00:00",
        "end": "2024-03-29T23:59:59"
    }
]

FRAME_INTERVAL_HOURS = 1
FRAMES_PER_SECOND = 6
GIF_DIMENSIONS = 720
OUTPUT_DIR = "meteosat_cloud_animations"

# Set these to numbers if auto-stretch is not good for your selected band.
VIS_MIN = None
VIS_MAX = None

METEOSAT_PALETTE = ["111111", "333333", "666666", "999999", "cccccc", "ffffff"]
PRECIP_PALETTE = ["000096", "0064ff", "00b4ff", "33db80", "9beb4a", "ffeb00", "ffb300", "ff6400", "eb1e00", "af0000"]

## 3. Earth Engine setup and AOI

In [ ]:
ee.Authenticate()
ee.Initialize(project=PROJECT_ID)

center = ee.Geometry.Point([MAROANTSETRA_LON, MAROANTSETRA_LAT])
aoi = center.buffer(AOI_BUFFER_KM * 1000).bounds()

Map = geemap.Map()
Map.centerObject(center, 8)
Map.addLayer(aoi, {"color": "yellow"}, "Animation AOI")
Map.addLayer(center, {"color": "red"}, "Maroantsetra")
Map

## 4. Select Meteosat collection and band

In [ ]:
def try_collection(collection_id, start, end):
    collection = ee.ImageCollection(collection_id).filterDate(start, end).filterBounds(aoi)
    count = collection.size().getInfo()
    first = ee.Image(collection.first())
    bands = first.bandNames().getInfo() if count else []
    return collection, count, bands


def select_collection_for_events():
    first_event = FLOOD_EVENTS[0]
    try:
        collection, count, bands = try_collection(
            METEOSAT_COLLECTION_ID,
            first_event["start"],
            first_event["end"]
        )
        print("Using Meteosat collection:", METEOSAT_COLLECTION_ID)
        print("Images in first event window:", count)
        print("Bands:", bands)
        return METEOSAT_COLLECTION_ID, False, bands
    except Exception as exc:
        if not USE_FALLBACK_PRECIP_IF_METEOSAT_UNAVAILABLE:
            raise
        print("Meteosat collection unavailable or empty for this account/window.")
        print("Falling back to Oya geostationary precipitation product.")
        print("Original error:", exc)
        collection, count, bands = try_collection(
            FALLBACK_COLLECTION_ID,
            first_event["start"],
            first_event["end"]
        )
        print("Fallback images in first event window:", count)
        print("Bands:", bands)
        return FALLBACK_COLLECTION_ID, True, bands


ACTIVE_COLLECTION_ID, USING_PRECIP_FALLBACK, available_bands = select_collection_for_events()

if CLOUD_BAND is not None:
    selected_band = CLOUD_BAND
else:
    selected_band = next((b for b in PREFERRED_CLOUD_BANDS if b in available_bands), available_bands[0])

print("Selected band:", selected_band)

## 5. Animation helpers

In [ ]:
def event_collection(event):
    return (
        ee.ImageCollection(ACTIVE_COLLECTION_ID)
        .filterDate(event["start"], event["end"])
        .filterBounds(aoi)
    )


def auto_vis_range(collection, band):
    if VIS_MIN is not None and VIS_MAX is not None:
        return VIS_MIN, VIS_MAX

    first = ee.Image(collection.select(band).first())
    stats = first.reduceRegion(
        reducer=ee.Reducer.percentile([2, 98]),
        geometry=aoi,
        scale=5000 if USING_PRECIP_FALLBACK else 5000,
        bestEffort=True,
        maxPixels=1e7
    ).getInfo()

    lo = stats.get(f"{band}_p2")
    hi = stats.get(f"{band}_p98")

    if lo is None or hi is None or lo == hi:
        if USING_PRECIP_FALLBACK:
            return 0, 15
        return 180, 320

    return lo, hi


def regular_frames(collection, event, band):
    start = ee.Date(event["start"])
    end = ee.Date(event["end"])
    n = end.difference(start, "hour").divide(FRAME_INTERVAL_HOURS).ceil().int()
    empty = ee.Image.constant(0).rename(band).updateMask(ee.Image(0))

    def make_frame(i):
        i = ee.Number(i)
        d0 = start.advance(i.multiply(FRAME_INTERVAL_HOURS), "hour")
        d1 = d0.advance(FRAME_INTERVAL_HOURS, "hour")
        subset = collection.filterDate(d0, d1).select(band)
        img = ee.Image(ee.Algorithms.If(subset.size().gt(0), subset.mosaic(), empty))
        return img.set({
            "system:time_start": d0.millis(),
            "label": d0.format("YYYY-MM-dd HH:mm")
        })

    return ee.ImageCollection(ee.List.sequence(0, n.subtract(1)).map(make_frame))


def render_event_gif(event):
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    collection = event_collection(event)
    count = collection.size().getInfo()
    if count == 0:
        print("No images for", event["name"])
        return None

    lo, hi = auto_vis_range(collection, selected_band)
    frames = regular_frames(collection, event, selected_band)

    palette = PRECIP_PALETTE if USING_PRECIP_FALLBACK else METEOSAT_PALETTE
    video_args = {
        "region": aoi,
        "dimensions": GIF_DIMENSIONS,
        "framesPerSecond": FRAMES_PER_SECOND,
        "min": lo,
        "max": hi,
        "palette": palette,
        "crs": "EPSG:3857"
    }

    safe_name = "".join(c if c.isalnum() else "_" for c in event["name"]).strip("_")
    out_gif = os.path.join(OUTPUT_DIR, f"{safe_name}.gif")
    url = frames.getVideoThumbURL(video_args)

    response = requests.get(url, timeout=120)
    response.raise_for_status()
    with open(out_gif, "wb") as f:
        f.write(response.content)

    print(event["name"])
    print("Raw images in event window:", count)
    print("Band:", selected_band, "min/max:", lo, hi)
    print("GIF:", out_gif)
    display(Image(filename=out_gif))
    return out_gif

## 6. Create animations

In [ ]:
gif_paths = []
for event in FLOOD_EVENTS:
    path = render_event_gif(event)
    if path:
        gif_paths.append(path)

display(pd.DataFrame({"gif": gif_paths}))

## 7. Inspect one event on an interactive map

In [ ]:
EVENT_INDEX = 0

inspect_event = FLOOD_EVENTS[EVENT_INDEX]
inspect_collection = event_collection(inspect_event).select(selected_band)
lo, hi = auto_vis_range(inspect_collection, selected_band)

Map = geemap.Map()
Map.centerObject(center, 8)
Map.addLayer(aoi, {"color": "yellow"}, "Animation AOI")
Map.addLayer(
    inspect_collection.first(),
    {"min": lo, "max": hi, "palette": PRECIP_PALETTE if USING_PRECIP_FALLBACK else METEOSAT_PALETTE},
    inspect_event["name"]
)
Map.addLayer(center, {"color": "red"}, "Maroantsetra")
Map

## Notes

- If the Meteosat SEVIRI collection is unavailable in your Earth Engine account, the notebook falls back to Oya precipitation. Oya is not raw Meteosat imagery, but it is derived from geostationary VIS/IR sensors including Meteosat-9/10 and is useful for flood-event context.
- Adjust `FLOOD_EVENTS`, `AOI_BUFFER_KM`, `FRAME_INTERVAL_HOURS`, and `CLOUD_BAND` as needed.
- Use infrared/thermal bands when available to see clouds at night as well as during the day.